# Learning NLP

NLP is a set of steops followed to build and end to end NLP software.

Consists of following steps
- Data Acquisiton
- Text preparation
    - Text cleanup
    - Basic preprocessing
    - Advance preprocessing
- Feature engineering
- Modelling
    - Model Building 
    - Evaluation
- Deployment
    - Deployment
    - Monitoring
    - Updating the model

### Difference btwn ML and NLP

1. ML
    - Here you make features to make prediction
    - Pros
        - Since you make features yourself, you can interpret the outcome, for eg you can explain why you got a spcific ans and why you got wrong prediction.
    - Cons
        - You need to have vast domain knowledge, you need to know which features are imp and which are not.
2. NLP
    - Here you just give the input and the machine will automatically make features and make prediction
    - Pros
        - Makes task easier, automatic feature generation
    - Cons
        - You loose interpretability as you are not doing manual feature selection.



### Text Preprocessing

- Lowercasing : as py is case sensitive we need to change every word to lowercase
- Removing unimportant tags (for eg, html tags)
- Removing URLs
- Removing Punctuations (removing symbols)
    - if you want to keep a symbol in that text, you can do so
- Chat word treatment (dealing with short chat words)
- Spelling correction

---

In [ ]:
import numpy as np
import pandas as pd
import re
import string
from autocorrect import Speller
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer
 
import warnings
warnings.filterwarnings('ignore')


---

In [2]:
df = pd.read_csv('datasets/IMDB Dataset.csv')
df.head(10)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
5,"Probably my all-time favorite movie, a story o...",positive
6,I sure would like to see a resurrection of a u...,positive
7,"This show was an amazing, fresh & innovative i...",negative
8,Encouraged by the positive comments about this...,negative
9,If you like original gut wrenching laughter yo...,positive


In [3]:
# changing to lowercase

df['review'] = df['review'].str.lower()
df['review'].head()

0    one of the other reviewers has mentioned that ...
1    a wonderful little production. <br /><br />the...
2    i thought this was a wonderful way to spend ti...
3    basically there's a family where a little boy ...
4    petter mattei's "love in the time of money" is...
Name: review, dtype: str

In [4]:
# removing html tags

# -----------------------------------------------------------------------
# making a function to remove html tags
import re
def remove_html(txt):
    pattern = re.compile('<.*?>')
    return pattern.sub(r' ', txt)
# -----------------------------------------------------------------------


df['review'] = df['review'].apply(remove_html)
df['review'].head()

0    one of the other reviewers has mentioned that ...
1    a wonderful little production.   the filming t...
2    i thought this was a wonderful way to spend ti...
3    basically there's a family where a little boy ...
4    petter mattei's "love in the time of money" is...
Name: review, dtype: str

In [5]:
# removing links

# -----------------------------------------------------------------------
# making a function to links
def remove_url(txt):
    pattern = re.compile(r'https?://\S+|www\.\S+')
    return pattern.sub(r' ', txt)
# -----------------------------------------------------------------------



In [6]:
# removing punctuations

import string
string.punctuation
exclude = string.punctuation

# -----------------------------------------------------------------------
def remove_punc(txt):
    for char in exclude:
        txt = txt.replace(char, ' ')
    return txt

# -----------------------------------------------------------------------

remove_punc('heii !!! This is Biswajeet')

'heii     This is Biswajeet'

The above function will take too much time for large datasets in NLP, use the code below

In [7]:
def remove_punc(txt):
    return txt.translate(str.maketrans('', '', exclude))

remove_punc('heii !!! This is Biswajeet')


'heii  This is Biswajeet'

In [8]:
# getting all the chat words from 'slang.txt' and storing it in a list

chat_words = {}
with open('slang.txt', 'r', encoding='utf-8') as f:
    for line in f:
        if '=' in line:
            abbr, full = line.split('=', 1)
            chat_words[abbr.strip()] = full.strip()

chat_words_list = list(chat_words.items())

print(len(chat_words))

163


In [9]:
# replacing chat words

def replace_chat_words(txt):
    temp = []
    for word in txt.split():
        if word.upper() in chat_words:
            temp.append(chat_words[word.upper()])
        else:
            temp.append(word)
    return ' '.join(temp)



print(replace_chat_words('btw hello there'))
print(replace_chat_words('gn hello there'))


By The Way hello there
Good Night hello there


In [10]:
# spelling correction
# there are other tools other thah the autocorrect for spelling errors
spell = Speller(lang='en')


In [11]:
# removing stopwords
from nltk.corpus import stopwords

def remove_stopwords(txt):
    temp = []

    for word in txt.split():
        if word in stopwords.words('english'):
            temp.append('')
        else:
            temp.append(word)
        
    temp2 = temp[:]
    temp.clear()
    return ' '.join(temp2)


remove_stopwords('Hi my name is Pradhan')



'Hi  name  Pradhan'

In [12]:
# handling emojis

import emoji

print(emoji.demojize('Hii this is me 😂😂😂 - ❤️'))

Hii this is me :face_with_tears_of_joy::face_with_tears_of_joy::face_with_tears_of_joy: - :red_heart:


There are two ways to deal with similar words, you can either use Stemming or you can also use Lemmatization.
- Stemming is fast, but will not be appropriate to show the output to the user
- Lemmatization also converts all the words to root words just like stemming, but it is very slow.

If you dont want to show the input to the user, use stemming. But if you want to show the input to the use use Lemmatization.

In [13]:
# tokenization & stemming
import nltk
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

nltk.download('stopwords')
nltk.download('punkt_tab')
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ashura\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ashura\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [14]:
# Consolidating all steps into one pass (Much faster!)
def preprocess_text(text):
    # 1. Emojis first (safest)
    import emoji
    text = emoji.demojize(text)
    
    # 2. Cleanup URLs and Punctuation
    text = remove_url(text)
    text = remove_punc(text)
    
    # 3. Expand Slang/Chat words
    text = replace_chat_words(text)
    
    # 4. Spelling correction (STILL SLOW - Only uncomment if you have time)
    # text = str(spell(text))

    # 5. Tokenization, Stopword Removal & Stemming (Combined for speed)
    tokens = []
    for word in text.split():
        if word not in stop_words:
            tokens.append(ps.stem(word))
    
    return tokens


# Apply to the dataframe
df['review'] = df['review'].apply(preprocess_text)
df['review'].head(10)

0    [one, review, mention, watch, 1, oz, episod, y...
1    [wonder, littl, product, film, techniqu, unass...
2    [thought, wonder, way, spend, tear, in, my, ey...
3    [basic, there, famili, littl, boy, jake, think...
4    [petter, mattei, love, tear, in, my, eye, mone...
5    [probabl, alltim, favorit, movi, stori, selfle...
6    [sure, would, like, see, resurrect, date, seah...
7    [show, amaz, fresh, innov, idea, 70, first, ai...
8    [encourag, posit, comment, film, look, forward...
9    [like, origin, gut, wrench, laughter, like, mo...
Name: review, dtype: object

In [15]:
df

,review,sentiment
0,"[one, review, mention, watch, 1, oz, episod, y...",positive
1,"[wonder, littl, product, film, techniqu, unass...",positive
2,"[thought, wonder, way, spend, tear, in, my, ey...",positive
3,"[basic, there, famili, littl, boy, jake, think...",negative
4,"[petter, mattei, love, tear, in, my, eye, mone...",positive
...,...,...
49995,"[thought, movi, right, good, job, wasnt, creat...",positive
49996,"[bad, plot, bad, dialogu, bad, act, idiot, dir...",negative
49997,"[cathol, taught, parochi, elementari, school, ...",negative
49998,"[im, go, disagre, previou, comment, side, malt...",negative


---

### Lemmatization Pipeline (df2)

Now we create a parallel pipeline using Lemmatization for comparison.

In [16]:
import nltk
from nltk.stem import WordNetLemmatizer
wordnet_lemmatizer = WordNetLemmatizer()

nltk.download('wordnet')
nltk.download('omw-1.4')


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ashura\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\ashura\AppData\Roaming\nltk_data...


True

In [17]:
# Create df2 by reloading the clean data
df2 = pd.read_csv('datasets/IMDB Dataset.csv')
df2['review'] = df2['review'].str.lower()

def preprocess_text2(text):
    # 1. Emojis and cleanup
    import emoji
    text = emoji.demojize(text)
    text = remove_url(text)
    text = remove_punc(text)
    text = replace_chat_words(text)
    
    # 2. Lemmatization + Stopwords removal
    tokens = [wordnet_lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words]
    
    return tokens

print("Processing df2 with Lemmatization...")
df2['review'] = df2['review'].apply(preprocess_text2)
df2.head(10)

Processing df2 with Lemmatization...


,review,sentiment
0,"[one, reviewer, mentioned, watching, 1, oz, ep...",positive
1,"[wonderful, little, production, br, br, filmin...",positive
2,"[thought, wonderful, way, spend, Tears, In, My...",positive
3,"[basically, there, family, little, boy, jake, ...",negative
4,"[petter, matteis, love, Tears, In, My, Eyes, m...",positive
5,"[probably, alltime, favorite, movie, story, se...",positive
6,"[sure, would, like, see, resurrection, dated, ...",positive
7,"[show, amazing, fresh, innovative, idea, 70, f...",negative
8,"[encouraged, positive, comment, film, looking,...",negative
9,"[like, original, gut, wrenching, laughter, lik...",positive


In [15]:
df

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. the filming t...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive
...,...,...
49995,i thought this movie did a down right good job...,positive
49996,"bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,i am a catholic taught in parochial elementary...,negative
49998,i'm going to have to disagree with the previou...,negative
